# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imamaparivash-insan/run.starter.notebook/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row represents one content item for one client on one report date in `fact_content_daily_performance`. The March 2026 partition contains 9,841,378 rows, and a duplicate check found no repeated combinations of `client_hash_id`, `content_hash_id`, and `report_date`.

**Time window:** March 1–31, 2026, used as the proposed development window. The partition's observed minimum and maximum report dates are March 1 and March 31, 2026.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Inspect the schema of the March 2026 partition.
# This shows the actual column names and their data types.
# It reads the schema rather than printing millions of rows.

query = f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

# Display the column names and types returned by DuckDB
result = con.sql(query).df()
display(result)


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features (maximum five candidate inputs)

The ranking decision is made after the feature window closes on March 15, 2026. Features must use information from March 1–15 only.

- `gsc_impressions` — search visibility during March 1–15; available after the feature window's reporting data has been collected.
- `gsc_clicks` — search clicks during March 1–15; available after that reporting data has been collected. March 16–22 clicks are reserved for the outcome.
- `gsc_avg_position` — reported average search position during March 1–15; available after the feature window's reporting data has been collected.
- `ga4_pageviews` — page-view activity during March 1–15; use only after checking how missing values and the availability flag should be interpreted.
- `ga4_engaged_sessions` — engaged-session activity during March 1–15; use only after checking its missingness and source availability.

These are candidate inputs, not yet validated predictors. Do not use values from after March 15 as features.

## Label (proposed outcome)

- `gsc_clicks` summed per client and content item for March 16–22, 2026, is the proposed future search-traffic outcome.
- The feature and outcome windows do not overlap, as verified by Query 3.
- March 16–22 `gsc_clicks` must never be used as a feature when predicting that same period's outcome. The outcome measures subsequent clicks, not whether a page definitively deserves refreshing.

## Context

- `report_date` — identifies the reporting date.
- `month` — identifies the monthly partition.
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — describe source availability and help interpret measurements.

## Excluded

- `client_hash_id`, `content_hash_id` — use for grouping and joining, not as model features.
- All March 16–22 outcome values — excluded from the feature set to prevent target leakage.
- GA4 measures with unresolved missingness or availability inconsistencies — do not assume missing values or zero values mean the same thing.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# ML-04 — QUERY 1: Verify the March 2026 data contract
# Checks the date range, total rows, and uniqueness of the proposed row grain.

# 1. Import the required libraries
from google.colab import userdata
import duckdb

# 2. Retrieve the Hugging Face token from Colab Secrets
# The token stays in Colab Secrets, not directly in this notebook.
HF_TOKEN = userdata.get("HF_TOKEN")

# 3. Create a DuckDB connection
con = duckdb.connect()

# 4. Configure Hugging Face authentication
# The token is passed as a parameter rather than inserted into SQL text.
con.execute(
    "CREATE SECRET hf_secret (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

# 5. Define the dataset location
dataset_path = "hf://datasets/FlyRank/internship-warehouse"

# 6. Define Query 1
# Group by client, content item, and report date to test the proposed row grain.
query = f"""
WITH grouped AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS rows_per_combination
    FROM read_parquet(
        '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
)
SELECT
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date,
    SUM(rows_per_combination) AS total_rows,
    COUNT(*) AS distinct_combinations,
    COUNT(*) FILTER (
        WHERE rows_per_combination > 1
    ) AS duplicate_combinations
FROM grouped
"""

# 7. Run Query 1 and display its results
result = con.sql(query).df()
display(result)





FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,total_rows,distinct_combinations,duplicate_combinations
0,2026-03-01,2026-03-31,9841378.0,9841378,0


In [ ]:

from google.colab import userdata

token = userdata.get("HF_TOKEN")
print("HF_TOKEN retrieved successfully:", bool(token))


HF_TOKEN retrieved successfully: True


In [ ]:


# ML-04 — QUERY 2: Check GA4 availability and missing pageviews

query = f"""
SELECT
    CAST(ga4_data_available AS VARCHAR) AS ga4_availability,
    COUNT(*) AS total_rows,
    COUNT(ga4_pageviews) AS rows_with_pageviews,
    COUNT(*) FILTER (
        WHERE ga4_pageviews IS NULL
    ) AS rows_with_missing_pageviews
FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY ga4_data_available
ORDER BY ga4_availability NULLS FIRST
"""

result = con.sql(query).fetchall()

print("GA4 availability | Total rows | Rows with pageviews | Rows with missing pageviews")
for row in result:
    print(row)



GA4 availability | Total rows | Rows with pageviews | Rows with missing pageviews
(None, 3018741, 0, 3018741)
('false', 6408671, 6408671, 0)
('true', 413966, 413966, 0)


In [ ]:

# ML-04 — QUERY 3: Verify non-overlapping feature and outcome windows

query = f"""
SELECT
    MAX(CASE
        WHEN report_date BETWEEN DATE '2026-03-01'
                             AND DATE '2026-03-15'
        THEN report_date
    END) AS feature_window_end,

    MIN(CASE
        WHEN report_date BETWEEN DATE '2026-03-16'
                             AND DATE '2026-03-22'
        THEN report_date
    END) AS outcome_window_start,

    COUNT(*) FILTER (
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-15'
    ) AS feature_window_rows,

    COUNT(*) FILTER (
        WHERE report_date BETWEEN DATE '2026-03-16'
                              AND DATE '2026-03-22'
    ) AS outcome_window_rows

FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

result = con.sql(query).fetchone()

print("Feature window ends:", result[0])
print("Outcome window starts:", result[1])
print("Feature window rows:", result[2])
print("Outcome window rows:", result[3])


Feature window ends: 2026-03-15
Outcome window starts: 2026-03-16
Feature window rows: 4642255
Outcome window rows: 2256218


In [ ]:

# ML-04 — Leakage check
# Confirm that the feature and outcome date windows do not overlap.

query = f"""
SELECT
    COUNT(*) FILTER (
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-15'
    ) AS feature_rows,

    COUNT(*) FILTER (
        WHERE report_date BETWEEN DATE '2026-03-16'
                              AND DATE '2026-03-22'
    ) AS outcome_rows,

    COUNT(*) FILTER (
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-15'
        AND report_date BETWEEN DATE '2026-03-16'
                            AND DATE '2026-03-22'
    ) AS overlapping_rows

FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

result = con.sql(query).fetchone()

print("Feature rows:", result[0])
print("Outcome rows:", result[1])
print("Overlapping rows:", result[2])


Feature rows: 4642255
Outcome rows: 2256218
Overlapping rows: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Verified data-availability limitation

In the March 2026 partition, `ga4_data_available` is `NULL` for 3,018,741 of 9,841,378 rows. A verification query confirmed that `ga4_pageviews` is also `NULL` for all these rows.

However, the availability flag and pageview values do not always agree: 6,408,671 rows have `ga4_data_available = false` while pageview values are present. Therefore, the flag must not be used alone to interpret pageview values. The reason for this inconsistency remains unverified.

### Evaluation limitation

The feature window (March 1–15) and outcome window (March 16–22) do not overlap. This prevents direct date overlap, but does not establish that every content item has complete history or that future clicks can be predicted reliably. The proposed outcome measures subsequent search clicks; it does not prove that a page should be refreshed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:

# Import userdata so we can safely access secrets stored in Google Colab
from google.colab import userdata

# Import DuckDB, which lets us query the dataset using SQL
import duckdb

# Retrieve the Hugging Face read token from Colab Secrets
# The secret is named HF_TOKEN, so we don't need to write the token here
HF_TOKEN = userdata.get("HF_TOKEN")

# Create a DuckDB connection so we can run SQL queries
con = duckdb.connect()

# Give DuckDB the token so it can access the gated Hugging Face dataset
# The actual token comes from Colab Secrets, not from the notebook source
con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Print a message to confirm that the secret was configured successfully
print("Connection setup complete!")


Connection setup complete!


In [ ]:

# Base location of the dataset on Hugging Face
dataset_path = "hf://datasets/FlyRank/internship-warehouse"

# Check that DuckDB can read the Parquet files for the daily performance table
query = f"""
SELECT *
FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5
"""

# Run the query and display five example rows
result = con.sql(query).df()
display(result)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [ ]:

# Check the date range and number of rows in the March 2026 partition.
# We use March as our development window, not the sealed June test month.

query = f"""
SELECT
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date,
    COUNT(*) AS total_rows
FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

# Execute the SQL query and display its single summary row
result = con.sql(query).df()
display(result)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,total_rows
0,2026-03-01,2026-03-31,9841378


In [ ]:




# Count how many client-content-date combinations appear more than once
query = f"""
SELECT COUNT(*) AS duplicate_combinations
FROM (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date
    FROM read_parquet(
        '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
) AS duplicates
"""

# Execute the query and print the result clearly
result = con.sql(query).fetchone()
print("Duplicate combinations:", result[0])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate combinations: 0


In [ ]:

# Check how many March rows have GSC and GA4 data available.
# This helps us understand whether missing data needs special handling.

query = f"""
SELECT
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS row_count
FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    gsc_data_available,
    ga4_data_available
ORDER BY
    gsc_data_available,
    ga4_data_available
"""

# Run the query and display the counts for each availability combination
result = con.sql(query).df()
display(result)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,ga4_data_available,row_count
0,False,False,4690323
1,False,True,49619
2,False,<NA>,1490375
3,True,False,1718348
4,True,True,364347
5,True,<NA>,1528366


In [ ]:

# Check how many March rows have GSC and GA4 data available.
# This helps us understand whether missing data needs special handling.

query = f"""
SELECT
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS row_count
FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    gsc_data_available,
    ga4_data_available
ORDER BY
    gsc_data_available,
    ga4_data_available
"""

# Run the query and display the counts for each availability combination
# Print the column names and each row without table formatting
result = con.sql(query).fetchall()

print("Columns:", [col[0] for col in con.description])

for row in result:
    print(row)

Columns: ['Success']
(False, False, 4690323)
(False, True, 49619)
(False, None, 1490375)
(True, False, 1718348)
(True, True, 364347)
(True, None, 1528366)


In [ ]:

# Investigate GA4 pageviews when the GA4 availability flag is missing.
# We count zero, positive, and NULL pageview values separately.

query = f"""
SELECT
    CASE
        WHEN ga4_pageviews IS NULL THEN 'NULL pageviews'
        WHEN ga4_pageviews = 0 THEN 'Zero pageviews'
        ELSE 'Positive pageviews'
    END AS pageview_status,
    COUNT(*) AS row_count
FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS NULL
GROUP BY pageview_status
ORDER BY pageview_status
"""

# Fetch and print the results clearly
result = con.sql(query).fetchall()

for row in result:
    print(row)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

('NULL pageviews', 3018741)


In [ ]:

# Define the ranking date and the future evaluation window.
# The feature window must end before the future outcome window begins.

query = f"""
SELECT
    MAX(CASE
        WHEN report_date BETWEEN DATE '2026-03-01'
                             AND DATE '2026-03-15'
        THEN report_date
    END) AS feature_window_end,

    MIN(CASE
        WHEN report_date BETWEEN DATE '2026-03-16'
                             AND DATE '2026-03-22'
        THEN report_date
    END) AS outcome_window_start,

    COUNT(*) FILTER (
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-15'
    ) AS feature_rows,

    COUNT(*) FILTER (
        WHERE report_date BETWEEN DATE '2026-03-16'
                              AND DATE '2026-03-22'
    ) AS outcome_rows

FROM read_parquet(
    '{dataset_path}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

# Run the check and print the result
result = con.sql(query).fetchone()
print(result)


(datetime.date(2026, 3, 15), datetime.date(2026, 3, 16), 4642255, 2256218)
